In [ ]:
from bs4 import BeautifulSoup
import requests
import json

In [ ]:
url = "https://www.bankofamerica.com/credit-cards/"



# headers = {
#     "User-Agent": "Mozilla/5.0"
# }
res = requests.get(url)
print(res)

soup = BeautifulSoup(res.text, "html.parser")
# print(f"here is yje sou{soup}")
txt = soup.get_text(" ",strip = True)

# print("______________________________________________________________________________________________")
# print(txt)



data = [
    {
        "source" : url,
        "content" : txt
    }
]

print(data)


with open("data/raw_data.json","w",encoding="utf-8") as cont:
    json.dump(data,cont,indent = 4)

<Response [200]>
[{'source': 'https://www.bankofamerica.com/credit-cards/', 'content': 'Credit Cards: Find & Apply for a Credit Card Online at Bank of America Skip to main content Bank of America Sign\xa0in Log in Locations En español Show/Hide Menu related links Bank of America Home Contact Us Locations Help Schedule an appointment En español En español Top Credit Card Links Top Credit Card Links Credit Card Home Application Center Check for Customized Offers Manage Your Credit Card Account View All Credit Cards Credit Card Home Application Center Check for Customized Offers Manage Your Credit Card Account View All Credit Cards Credit Cards by Category Credit Cards by Category Cash Back Credit Cards Credit Cards for Building Credit Credit Cards for Students Lower Interest Rate Credit Cards Points Rewards Credit Cards Travel Rewards Credit Cards Airline Rewards Credit Cards Cash Back Credit Cards Credit Cards for Building Credit Credit Cards for Students Lower Interest Rate Credit Card

In [ ]:
from langchain_core.documents import Document
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [1]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin

BASE_URL = "https://www.bankofamerica.com"
START_URL = "https://www.bankofamerica.com/credit-cards/"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


def get_soup(url):
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


# -------------------------
# Step 1 : Get Landing Page
# -------------------------

landing_soup = get_soup(START_URL)

# -------------------------
# Step 2 : Find Product Links
# -------------------------

card_links = set()

for link in landing_soup.find_all("a", href=True):

    href = link["href"]

    if "/credit-cards/" in href and "apply" not in href:

        full_url = urljoin(BASE_URL, href)

        card_links.add(full_url)

print(f"Found {len(card_links)} pages")

# -------------------------
# Step 3 : Visit each page
# -------------------------

cards = []

for url in sorted(card_links):

    try:

        print("Scraping:", url)

        soup = get_soup(url)

        # --------------------
        # Card Name
        # --------------------

        h1 = soup.find("h1")

        card_name = h1.get_text(strip=True) if h1 else "Unknown"

        # --------------------
        # Category
        # --------------------

        category = ""

        breadcrumbs = soup.find_all("li")

        if breadcrumbs:
            category = breadcrumbs[-2].get_text(strip=True) if len(breadcrumbs) > 1 else ""

        # --------------------
        # Features
        # --------------------

        features = []

        for li in soup.find_all("li"):

            text = li.get_text(" ", strip=True)

            if len(text) > 30:
                features.append(text)

        # remove duplicates

        features = list(dict.fromkeys(features))

        # --------------------
        # Description
        # --------------------

        description = ""

        p = soup.find("p")

        if p:
            description = p.get_text(" ", strip=True)

        # --------------------
        # Benefits
        # --------------------

        benefits = " ".join(features[:5])

        # --------------------
        # Rates & Fees
        # --------------------

        rates = []

        page_text = soup.get_text(" ", strip=True)

        keywords = [
            "APR",
            "annual fee",
            "balance transfer",
            "interest rate",
            "foreign transaction fee"
        ]

        sentences = page_text.split(".")

        for sentence in sentences:

            for key in keywords:

                if key.lower() in sentence.lower():

                    rates.append(sentence.strip())

                    break

        rates = list(dict.fromkeys(rates))

        cards.append(
            {
                "card_name": card_name,
                "category": category,
                "description": description,
                "features": features,
                "benefits": benefits,
                "rates_fees": rates,
                "source": url,
            }
        )

    except Exception as e:

        print("Skipped:", url)
        print(e)

# -------------------------
# Save JSON
# -------------------------

with open("data/raw_data.json", "w", encoding="utf-8") as f:

    json.dump(cards, f, indent=4, ensure_ascii=False)

print("Done")

Found 19 pages
Scraping: https://www.bankofamerica.com/banking-information/assistance/credit-cards/credit-cards-assistance-overview.go?request_locale=en_US
Scraping: https://www.bankofamerica.com/credit-cards/
Scraping: https://www.bankofamerica.com/credit-cards/airline-credit-cards/
Scraping: https://www.bankofamerica.com/credit-cards/application-status-center/
Scraping: https://www.bankofamerica.com/credit-cards/cash-back-credit-cards/
Scraping: https://www.bankofamerica.com/credit-cards/compare-credit-cards/
Scraping: https://www.bankofamerica.com/credit-cards/credit-card-account-information-faq/
Scraping: https://www.bankofamerica.com/credit-cards/credit-card-agreements/
Scraping: https://www.bankofamerica.com/credit-cards/credit-card-with-no-annual-fee/
Scraping: https://www.bankofamerica.com/credit-cards/credit-cards-to-build-credit/
Scraping: https://www.bankofamerica.com/credit-cards/low-interest-credit-cards/
Scraping: https://www.bankofamerica.com/credit-cards/manage-your-cre

In [2]:
import os
import json
import asyncio
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from crawl4ai import AsyncWebCrawler, CacheMode
from crawl4ai.async_configs import CrawlerRunConfig
from crawl4ai.extraction_strategy import LLMExtractionStrategy, create_llm_config

# Self-correcting schema rules passed directly to Groq
class CreditCardSchema(BaseModel):
    card_name: str = Field(..., description="Official proper name of the card. Return 'SKIP' if not a card profile.")
    category: str = Field(..., description="Travel Rewards, Cash Back, Balance Transfer, Student, etc.")
    features: str = Field(..., description="Core reward multipliers or key earning structures.")
    benefits: str = Field(..., description="Sign-up offers, statement credits, or perks.")
    rates_fees: str = Field(..., description="Annual fee details and APR specifications.")
    product_url: str = Field(..., description="Hyperlink string if available, else return 'Link in Main Catalog'.")

async def fetch_raw_markdown(url: str) -> str:
    """Uses Crawl4AI to parse layout content strings down cleanly."""
    async with AsyncWebCrawler(verbose=False) as crawler:
        result = await crawler.arun(
            url=url, 
            config=CrawlerRunConfig(cache_mode=CacheMode.BYPASS, excluded_tags=["nav", "footer"])
        )
    if not result.success:
        raise RuntimeError(f"Crawl operations failed: {result.error_message}")
    return result.markdown.fit_markdown or result.markdown.raw_markdown

async def extract_card_properties_from_chunk(chunk_text: str, source_label: str, chunk_idx: int) -> List[Dict[str, Any]]:
    """Runs a single preprocessed metadata slice securely through the Groq extraction layer."""
    groq_key = os.environ.get("GROQ_API_KEY")
    if not groq_key:
        raise ValueError("CRITICAL: Environment variable GROQ_API_KEY is missing.")
        
    llm_config = create_llm_config(provider="groq/llama-3.3-70b-versatile", api_token=groq_key)
    strategy = LLMExtractionStrategy(
        llm_config=llm_config, 
        extraction_type="schema",
        schema=CreditCardSchema.model_json_schema(), 
        input_format="fit_markdown",
        instruction="Extract all listed credit card offers into structured JSON matching the schema.", 
        verbose=False
    )
    
    async with AsyncWebCrawler(verbose=False) as crawler:
        safe_virtual_url = f"raw://data_pipeline/{source_label}/chunk_{chunk_idx}/{chunk_text}"
        result = await crawler.arun(
            url=safe_virtual_url, 
            config=CrawlerRunConfig(extraction_strategy=strategy, cache_mode=CacheMode.BYPASS)
        )
        
    if result.success and result.extracted_content:
        try:
            parsed = json.loads(result.extracted_content)
            return parsed if isinstance(parsed, list) else [parsed]
        except json.JSONDecodeError:
            return []
    return []

In [5]:
import os
import json
import asyncio
from typing import List, Dict, Any

from pydantic import BaseModel, Field

from crawl4ai import AsyncWebCrawler, CacheMode
from crawl4ai.async_configs import CrawlerRunConfig
from crawl4ai.extraction_strategy import (
    LLMExtractionStrategy,
    create_llm_config,
)


# -----------------------------
# Schema
# -----------------------------
class CreditCardSchema(BaseModel):
    card_name: str = Field(
        ..., description="Official card name. Return SKIP if not a credit card."
    )
    category: str = Field(
        ..., description="Travel Rewards, Cash Back, Student, etc."
    )
    features: str = Field(
        ..., description="Main reward earning features."
    )
    benefits: str = Field(
        ..., description="Bonuses and benefits."
    )
    rates_fees: str = Field(
        ..., description="APR and annual fee information."
    )
    product_url: str = Field(
        ..., description="Product page URL if available."
    )


# -----------------------------
# Fetch webpage as markdown
# -----------------------------
async def fetch_markdown(url: str) -> str:

    async with AsyncWebCrawler(verbose=True) as crawler:

        result = await crawler.arun(
            url=url,
            config=CrawlerRunConfig(
                cache_mode=CacheMode.BYPASS,
                excluded_tags=["nav", "footer"]
            ),
        )

    if not result.success:
        raise Exception(result.error_message)

    return result.markdown.fit_markdown or result.markdown.raw_markdown


# -----------------------------
# Extract structured information
# -----------------------------
async def extract_cards(markdown: str):

    groq_api = os.getenv("GROQ_API_KEY")

    if not groq_api:
        raise Exception("GROQ_API_KEY not found.")

    llm_config = create_llm_config(
        provider="groq/llama-3.3-70b-versatile",
        api_token=groq_api,
    )

    strategy = LLMExtractionStrategy(
        llm_config=llm_config,
        extraction_type="schema",
        schema=CreditCardSchema.model_json_schema(),
        input_format="markdown",
        instruction="""
Extract every Bank of America credit card.

Return a JSON array.

For each card extract:

- card_name
- category
- features
- benefits
- rates_fees
- product_url

Ignore navigation menus, advertisements and unrelated content.
""",
        verbose=True,
    )

    async with AsyncWebCrawler(verbose=True) as crawler:

        result = await crawler.arun(
            url=f"raw://{markdown}",
            config=CrawlerRunConfig(
                extraction_strategy=strategy,
                cache_mode=CacheMode.BYPASS,
            ),
        )

    if not result.success:
        raise Exception(result.error_message)

    cards = json.loads(result.extracted_content)

    if isinstance(cards, dict):
        cards = [cards]

    return cards


# -----------------------------
# Main
# -----------------------------
async def main():

    url = "https://www.bankofamerica.com/credit-cards/"

    print("Fetching webpage...")

    markdown = await fetch_markdown(url)

    print("Extracting structured data using Groq...")

    cards = await extract_cards(markdown)

    os.makedirs("data", exist_ok=True)

    with open(
        "data/raw_data.json",
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            cards,
            f,
            indent=4,
            ensure_ascii=False,
        )

    print(f"\nSaved {len(cards)} credit cards to data/raw_data.json")

await main()

Fetching webpage...


NotImplementedError: 